# Telco Customer Churn: EDA and Data Cleaning

This notebook profiles the source data, cleans `TotalCharges` and service categories, and verifies the model feature schema. The reproducible split, scaling, training, and evaluation workflow is in `ML/training/train_and_evaluate.py`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

data_candidates = [
    Path("ML/data/WA_Fn-UseC_-Telco-Customer-Churn.csv"),
    Path("data/WA_Fn-UseC_-Telco-Customer-Churn.csv"),
    Path("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"),
]
data_path = next((path for path in data_candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("Could not locate the Telco churn CSV in ML/data.")

df_raw = pd.read_csv(data_path)
print(f"Loaded {len(df_raw):,} rows and {df_raw.shape[1]} columns from {data_path}")
df_raw.head()

In [ ]:
display(df_raw.describe(include="all").T)
quality = pd.DataFrame({
    "missing": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean() * 100).round(2),
    "unique_values": df_raw.nunique(dropna=False),
})
display(quality.sort_values("missing", ascending=False))
print("Exact duplicate rows:", int(df_raw.duplicated().sum()))
print("Blank TotalCharges values:", int(df_raw["TotalCharges"].astype(str).str.strip().eq("").sum()))

In [ ]:
df = df_raw.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]
for column in service_cols:
    df[column] = df[column].replace("No internet service", "No")
df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype("int8")
df = df.drop(columns=["customerID"], errors="ignore")

print("Rows with missing values after cleaning:", int(df.isna().any(axis=1).sum()))
display(df.head())
display(df["Churn"].value_counts(normalize=True).rename("share"))

In [ ]:
display(df.groupby("Contract", observed=True)["Churn"].mean().sort_values(ascending=False))
display(df.groupby("InternetService", observed=True)["Churn"].mean().sort_values(ascending=False))
display(df.groupby("PaymentMethod", observed=True)["Churn"].mean().sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="Churn", y="tenure", ax=axes[0])
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
axes[0].set_title("Tenure by churn outcome")
axes[1].set_title("Monthly charges by churn outcome")
plt.tight_layout()

In [ ]:
feature_names = [
    "SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges",
    "gender_Male", "Partner_Yes", "Dependents_Yes", "PhoneService_Yes",
    "MultipleLines_Yes", "InternetService_Fiber optic", "InternetService_No",
    "OnlineSecurity_Yes", "OnlineBackup_Yes", "DeviceProtection_Yes",
    "TechSupport_Yes", "StreamingTV_Yes", "StreamingMovies_Yes",
    "Contract_One year", "Contract_Two year", "PaperlessBilling_Yes",
    "PaymentMethod_Credit card (automatic)",
    "PaymentMethod_Electronic check", "PaymentMethod_Mailed check",
]
X = pd.get_dummies(df.drop(columns=["Churn"]), drop_first=True, dtype=np.int8)
X = X.reindex(columns=feature_names, fill_value=0)
y = df["Churn"]
assert X.shape == (len(df), 23)
assert X.columns.tolist() == feature_names
print("Verified model feature matrix:", X.shape)
print("Feature order:", X.columns.tolist())